# My Version

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset # Gunakan Subset
...
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split # Tambahkan ini
import numpy as np

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 3. Load Dataset & Stratified Splitting
data_dir = '/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color'
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Dapatkan semua label untuk keperluan stratifikasi
labels = full_dataset.targets

# Lakukan Stratified Split menggunakan sklearn (80% Train, 20% Val)
train_indices, val_indices = train_test_split(
    range(len(labels)),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# Buat Subset dataset berdasarkan indeks yang sudah terstratifikasi
train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

# 4. Data Loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Total Images: {len(full_dataset)}")
print(f"Training Images: {len(train_dataset)} (Stratified)")
print(f"Validation Images: {len(val_dataset)} (Stratified)")

# 5. Build Model (EfficientNet-B0)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, len(full_dataset.classes))
model = model.to(device)

# 6. Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 7. Training & Validation Loop
num_epochs = 5

for epoch in range(num_epochs):
    # --- PHASE TRAINING ---
    model.train()
    train_loss = 0.0
    t_preds, t_labels = [], []
    
    pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", unit="batch")
    for images, labels in pbar_train:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, pred = torch.max(outputs, 1)
        t_preds.extend(pred.cpu().numpy()); t_labels.extend(labels.cpu().numpy())
        
        curr_f1 = f1_score(t_labels, t_preds, average='macro')
        pbar_train.set_postfix({"loss": f"{loss.item():.3f}", "f1": f"{curr_f1:.3f}"})

    # --- PHASE VALIDATION ---
    model.eval()
    val_loss = 0.0
    v_preds, v_labels = [], []
    
    print(f"Menjalankan Validasi Epoch {epoch+1}...")
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, pred = torch.max(outputs, 1)
            v_preds.extend(pred.cpu().numpy()); v_labels.extend(labels.cpu().numpy())

    final_train_f1 = f1_score(t_labels, t_preds, average='macro')
    final_val_f1 = f1_score(v_labels, v_preds, average='macro')
    
    print(f"\n>> Epoch {epoch+1} Selesai")
    print(f"Train Loss: {train_loss/len(train_dataset):.4f} | Train F1: {final_train_f1:.4f}")
    print(f"Val Loss: {val_loss/len(val_dataset):.4f} | Val F1: {final_val_f1:.4f}\n")

# 8. Save Model
torch.save(model.state_dict(), 'efficientnet_b0_stratified.pth')
print("Model Saved!")

Total Images: 54305
Training Images: 43444 (Stratified)
Validation Images: 10861 (Stratified)


Epoch 1/5 [Train]:   0%|          | 0/1358 [00:00<?, ?batch/s]

Menjalankan Validasi Epoch 1...

>> Epoch 1 Selesai
Train Loss: 0.3897 | Train F1: 0.8871
Val Loss: 0.0334 | Val F1: 0.9853



Epoch 2/5 [Train]:   0%|          | 0/1358 [00:00<?, ?batch/s]

Menjalankan Validasi Epoch 2...

>> Epoch 2 Selesai
Train Loss: 0.0402 | Train F1: 0.9841
Val Loss: 0.0215 | Val F1: 0.9914



...

Menjalankan Validasi Epoch 3...

>> Epoch 3 Selesai
Train Loss: 0.0236 | Train F1: 0.9899
Val Loss: 0.0147 | Val F1: 0.9929



Epoch 4/5 [Train]:   0%|          | 0/1358 [00:00<?, ?batch/s]

Menjalankan Validasi Epoch 4...

>> Epoch 4 Selesai
Train Loss: 0.0149 | Train F1: 0.9936
Val Loss: 0.0167 | Val F1: 0.9926



Epoch 5/5 [Train]:   0%|          | 0/1358 [00:00<?, ?batch/s]

Menjalankan Validasi Epoch 5...

>> Epoch 5 Selesai
Train Loss: 0.0127 | Train F1: 0.9946
Val Loss: 0.0165 | Val F1: 0.9929

Model Saved!


In [6]:
import os
file_path = 'efficientnet_b0_stratified.pth'
if os.path.exists(file_path):
    print(f"File ditemukan! Ukuran: {os.path.getsize(file_path) / (1024*1024):.2f} MB")
else:
    print("File TIDAK ditemukan. Cek kembali path penyimpanan Anda.")

File ditemukan! Ukuran: 15.76 MB


In [7]:
!zip -r models_backup.zip /kaggle/working/

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 71%)
  adding: kaggle/working/efficientnet_b0_stratified.pth (deflated 8%)


In [8]:
from IPython.display import FileLink
# Ganti nama file sesuai dengan file zip Anda
FileLink(r'models_backup.zip')

/kaggle/working/models_backup.zip